## Music Recommendation
### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: March 18, 2026


---

#### Instructions

In this assignment, you will build an ALS recommendation algorithm based on user listening data from Autoscrobbler.

The data consists of: 
- user data (listeners)
- item data (songs)
- interaction data (user listens, which is implicit feedback).  

Additional details are in the `README` file. Be sure to review in advance.

Develop/test code to accomplish the tasks below, clearly showing the results and documenting your work.

You will use the **DataFrame API** for all work.

**TOTAL POINTS: 10**
***

#### Preprocessing Steps in order:

1) To reduce runtime, you will sample 20% of the interaction data from `user_artist_data.txt` using seed=314 (and use this seed throughout the lab)
2) Next, remove users that listened to fewer than 10 **artists** (you may ignore aliasing in this step)
3) Lastly, split the data: 60% for training / 20% for validation / 20% for testing

In [1]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 1
# To reduce runtime, you will sample 20% of the interaction data from user_artist_data.txt using seed=314 (and use this seed throughout the lab)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Before doing anything, create a Spark session.
# I added the .config settings per recommendation from Gemini to help reduce performance issues I was experiencing.
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("AutoscrobblerALS") \
    .config("spark.driver.memory", "32g") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

# Define the schema for the file to import.  
# -- I added this after reviewing the data file and running an initial import.
# -- Even though I saw the file had no headers, I didn't initially stop to think I needed to add them!  :)
from pyspark.sql.types import StructType, StructField, IntegerType
autoscrobbler_schema = StructType([
                                    StructField("userid", IntegerType()),
                                    StructField("artistid", IntegerType()),
                                    StructField("playcount", IntegerType()),
                                    ])
     

# Load the data.
# The columns are separated by a space, so need to include sep=" ".
df_user = spark.read.csv("/standard/ds7200-apt4c/large_datasets/user_artist_data.txt",  sep=" ", header=False, schema=autoscrobbler_schema)

# Check to see if any columns have null values.
df_user.createOrReplaceTempView("df_user")
df_user_null = spark.sql("""
    SELECT COUNT(*) AS RowsWithNulls
    FROM df_user
    WHERE 
        userid IS NULL 
        OR artistid IS NULL
        OR playcount IS NULL
""")
print(f"""There are {df_user_null.show()} rows with NULL values.""")

# Since no rows have null values, continue...
# Select a 20% sample from the data.
df_user_small = df_user.sample(fraction=0.20, seed=314)

# Confirm rows were imported.
print(f"""There are {df_user_small.count()} rows in the imported data set.""")

# Review a few records and the schema to make sure it was applied correctly.
# And it does, so yay!
df_user.show(5)
df_user.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/apps/software/standard/core/pyspark/4.2.0-py3.14/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/24 20:29:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

+-------------+
|RowsWithNulls|
+-------------+
|            0|
+-------------+

There are None rows with NULL values.
There are 4856055 rows in the imported data set.
+-------+--------+---------+
| userid|artistid|playcount|
+-------+--------+---------+
|1000002|       1|       55|
|1000002| 1000006|       33|
|1000002| 1000007|        8|
|1000002| 1000009|      144|
|1000002| 1000010|      314|
+-------+--------+---------+
only showing top 5 rows
root
 |-- userid: integer (nullable = true)
 |-- artistid: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [2]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 2
# Next, remove users that listened to fewer than 10 artists (you may ignore aliasing in this step)
# ----------------------------------------------------------------------------------------------------------------------------------------------

# Remove users that listen to fewer than 10 artists using SQL.
# -- First register the DF as a temp view first.
df_user_small.createOrReplaceTempView("df_user_small")

# See how may users have listened to fewer than 10 artists.

# Select the columns needed for this analysis.
df_user_go = spark.sql("""
    SELECT *
    FROM df_user_small
    WHERE userid IN
        (
            SELECT userid
            FROM df_user_small
            GROUP BY userid
            HAVING COUNT(DISTINCT artistid) > 9
        )
""")

print(f"""There are {df_user_go.count()} rows after removing users that listened to less than 10 artists.""")

[Stage 14:===========================================>              (6 + 2) / 8]

There are 4679612 rows after removing users that listened to less than 10 artists.


In [3]:
# ----------------------------------------------------------------------------------------------------------------------------------------------
# PREPROCESSING STEP 3
# Lastly, split the data: 60% for training / 20% for validation / 20% for testing
# ----------------------------------------------------------------------------------------------------------------------------------------------
# Split the dataset.
trainDF, valDF, testDF = df_user_go.randomSplit([.6, .2, .2], seed=314)

# Cache the datasets for better downstream processing.
trainDF = trainDF.cache()
valDF = valDF.cache()
testDF = testDF.cache()

print(f"""There are {trainDF.count()} rows in the training set, {valDF.count()} rows in the validation set, and {testDF.count()} in the test set.""")

[Stage 55:===========================================>            (25 + 7) / 32]

There are 2805992 rows in the training set, 937525 rows in the validation set, and 936095 in the test set.


In [4]:
# TASK 1 (1 PT)
# Show the first row from the training set
trainDF.show(1)

# Result should remain consistent with seed=314.
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   536|      30|        5|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   536|      82|       28|
+------+--------+---------+
only showing top 1 row


In [5]:
# TASK 2 (1 PT)
# Show the first row from the test set. We will refer to this user as Test User 1.
testDF.show(1)

# Result should remain consistent with seed=314.
#  +------+--------+---------+
#  |userid|artistid|playcount|
#  +------+--------+---------+
#  |   536|      30|        5|
#  +------+--------+---------+

+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   536|    1205|        2|
+------+--------+---------+
only showing top 1 row


In [6]:
# ALS MODEL 1: Train the ALS implicit model on the training data (since the measurements are activity and not ratings)
# Use parameters: rank=8, maxIter=5, regParam=0.02, alpha=0.01

# The parameters above do not align to the old MLlib library, so using the newer ML library.
from pyspark.ml.recommendation import ALS

# Create the model specification using the parameters for ML.
als_spec = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    coldStartStrategy="drop",
    rank=8,
    maxIter=5,
    regParam=0.02,
    alpha=0.01
)

# Fit the model.
als_model = als_spec.fit(trainDF)

print("The model has been trained.")

The model has been trained.


In [10]:
# TASK 3 (2 PTS)
# Using ALS MODEL 1, make 10 recommendations for Test User 1
from pyspark.sql import functions as F

als_predictions = als_model.transform(testDF)

# Use SQL to get 10 predictions for Test User 1 (userid = 536).
als_predictions.createOrReplaceTempView("als_predictions")

#---------------------------------------------------------------------------
# STEP 1: First, get the predictions for userid = 536.
#---------------------------------------------------------------------------
test_user1 = spark.sql("""
    SELECT *
    FROM als_predictions
    WHERE 
        userid=536
        AND NOT ISNAN(prediction)
""")

# The below was my submitted code which was wrong...I misunderstood the ask.
#---------------------------------------------------------------------------
# STEP 2: Then, get the top 10 predictions.
#---------------------------------------------------------------------------
# test_user1.createOrReplaceTempView("test_user1")
# test_user1_top10 = spark.sql("""
#     SELECT *
#     FROM test_user1
#     ORDER BY prediction DESC
#     LIMIT 10
# """)
#---------------------------------------------------------------------------

# This is the corrected code, which I think is right now per help from Gemini.
#---------------------------------------------------------------------------
# STEP 2: Then, generate the top 10 recommendations across all artists.
#---------------------------------------------------------------------------
user1_recs = als_model.recommendForUserSubset(test_user1, 10)

# Explode the nested recommendations array into clean rows.
test_user1_top10 = user1_recs.select(
                                        F.col("userid"),
                                        F.explode(F.col("recommendations")).alias("rec")
                                    ).select(
                                                F.col("userid"),
                                                F.col("rec.artistid").alias("artistid"),
                                                F.col("rec.rating").alias("prediction")
                                            )

test_user1_top10.show()

+------+--------+-----------+
|userid|artistid| prediction|
+------+--------+-----------+
|   536| 1000113| 0.26242226|
|   536|      82| 0.21813422|
|   536|    1274| 0.20020731|
|   536|     189| 0.19191323|
|   536| 1000024| 0.19177008|
|   536|     352| 0.15755582|
|   536|     873| 0.13393518|
|   536|     121|  0.1238538|
|   536|     313|0.114319466|
|   536|    3402| 0.11321966|
+------+--------+-----------+



**TASK 4 (1 PT)**  
**Briefly explain Recall@K in the context of recommender systems**

Recall@k is like the concept of recall in class imbalance problems which is the number of True Positives (TP) that are correctly predicted, e.g., TP / (TP + FN).  What's different for recommendation problems is that the equivalent of True Positives (the numerator) is the top number (k) of the total predicted items of which the user did interact with and which the model correctly predicted as such, and the denominator is this number plus all the other items that the user interacted with.

In [13]:
# TASK 5 (1 PT)
# Calculate Recall@K for Test User 1

# This is asking, "Of all the artists that the user listened to, how many of the top-k predictions did the model correctly predict?"
# Mathematically, the formula is:
#
#              # artists the user listened to that are predicted in the top k
#  Recall@K = ----------------------------------------------------------------
#                     # all of the artists the user listened to


# One way to do this is with SQL.
# I'll stick with k=10 for this.

# This requires, querying the test data.
testDF.createOrReplaceTempView("testDF")

# STEP 1: DENOMINATOR.
# Get the total rows for userid = 536.
all_items = spark.sql("""
SELECT *
FROM testDF
WHERE userid = 536
""")

# STEP 2: NUMERATOR.
# Get the total of these that were in the predicted set; these are the true positives (tp).
# This part uses the top k=10 from TASK 3.
# Use DISTINCT to get unique artists.
all_items.createOrReplaceTempView("all_items")
test_user1_top10.createOrReplaceTempView("test_user1_top10")
tp = spark.sql("""
SELECT DISTINCT *
FROM test_user1_top10 AS t
JOIN all_items AS a
ON t.artistid = a.artistid
""")

# STEP 3: CALCULATE the Recall@K.
tp.createOrReplaceTempView("tp")
recall_k = spark.sql("""
SELECT 
    ROUND(
            (SELECT COUNT(*) FROM tp) /
            (SELECT COUNT(*) FROM all_items)
            ,4
        ) AS RecallK
""")

# I had trouble extracting the value from the recall_k DataFrame so I had MS Copilot give me this syntax.
# I just don't do enough straight up Python coding for indexing to have sunk in.
print(f"The Recall@K for userid=536 is: {recall_k.collect()[0]['RecallK']}")

The Recall@K for userid=536 is: 0.0


**Recall@K for my model at this point.**  
The above results show that the Recall@K for userid=536 is: 0.0
The interpretation here is that out of all the artists listened to by userid=536 (in the test set), the modeled predicted none of these artists in its top 10 recommendations.

The next code cell checks to see if this result makes sense, because it seems odd at first glance.

In [15]:
# Verify if the result above makes sense.

print("Top 10 recommended artists for 536:")
test_user1_top10.show()

print("Test set artists for 536 (count):", all_items.count())
all_items.show(10)

print("Matching artists (true positives):", tp.count())

Top 10 recommended artists for 536:
+------+--------+-----------+
|userid|artistid| prediction|
+------+--------+-----------+
|   536| 1000113| 0.26242226|
|   536|      82| 0.21813422|
|   536|    1274| 0.20020731|
|   536|     189| 0.19191323|
|   536| 1000024| 0.19177008|
|   536|     352| 0.15755582|
|   536|     873| 0.13393518|
|   536|     121|  0.1238538|
|   536|     313|0.114319466|
|   536|    3402| 0.11321966|
+------+--------+-----------+

Test set artists for 536 (count): 13
+------+--------+---------+
|userid|artistid|playcount|
+------+--------+---------+
|   536|    1205|        2|
|   536|    1598|        2|
|   536|    4449|       30|
|   536| 1012701|       17|
|   536| 1025289|       11|
|   536| 1037206|        2|
|   536| 1173467|       32|
|   536| 1235135|       29|
|   536| 1339526|        1|
|   536| 2040459|       13|
+------+--------+---------+
only showing top 10 rows
Matching artists (true positives): 0


**Confirming the results.**  

These results confirm that the calculation is working properly...the model returned 10 top recommendations for userid = 536 based upon the training data.  However, of the 13 artist records in the test data set for userid = 536, none of them (9) overlap between the two sets.  Thus, 0 / 13 = 0.0.

In [16]:
# ALS MODEL 2: For this model, you will use hyperparameter tuning on the validation set to select the best values for
# (rank, maxIter, regParam) to maximize Recall@K. Fix alpha=0.01. Fit each candidate ALS model on the training data and evaluate its recommendations on the validation set.

# Import the necessary library.
from pyspark.ml.tuning import ParamGridBuilder

# Build the hyperparameter tuning grid.
# I had Gemini help recommend values for these parameters.
# I added comments to explain my understanding of why to use these values.
paramGrid = (
        ParamGridBuilder()
    
        # Specify different number of hidden features to look for.
        .addGrid(als_spec.rank, [10, 20, 40])
    
        # Specify how many times to loop through the data.
        .addGrid(als_spec.maxIter, [10, 20])
    
        # Specify the regularization penalty; 0.01 is flexible and 0.1 is stricter (less-overfitting).
        .addGrid(als_spec.regParam, [0.01, 0.1])
        
        # Build the tuning grid.
        .build()
    )

print("The paramGrid has been built.")

The paramGrid has been built.


**Concerning alpha.**  
I already have alpha=0.01 in the definition of `als_spec` earlier in ALS Model 1.  Thus, I don't need to re-specify this for the next step.

In [22]:
# TASK 6 (2 PTS)
# Report the best values for (rank, maxIter, regParam) along with the Recall@K from the validation set

# Import libraries.
from pyspark.sql import functions as F
import time

# Get unique validation users so we only generate recommendations for users in valDF.
val_users = valDF.select("userid").distinct()

# Get unique (userid, artistid) pairs in valDF as the ground-truth set.
actual_val = valDF.select("userid", "artistid").distinct()

# Denominator: total unique artists listened to per validation user.
val_totals = actual_val.groupBy("userid").agg(F.count("*").alias("total"))

# Track the highest validation Recall@10 and its winning hyperparameters.
best_recall, best_rank, best_maxIter, best_regParam = -1.0, None, None, None

# Start timer to track processing time.
t0 = time.time()

# Loop through every hyperparameter combination in paramGrid.
for p in paramGrid:
  # Fit the candidate ALS model on the training set (not valDF).
  model = als_spec.copy(p).fit(trainDF)

  # Generate top-10 artist recommendations for validation users, explode into rows,
  # and match against actual_val to count true positives (hits) per user.
  recs = (
      model.recommendForUserSubset(val_users, 10)
      .select("userid", F.explode("recommendations.artistid").alias("artistid"))
      .join(actual_val, ["userid", "artistid"])
      .groupBy("userid")
      .agg(F.count("*").alias("hits"))
  )

  # Calculate per-user Recall@10 (0 if a user had no hits) and average across users.
  row = (
      val_totals.join(recs, "userid", "left")
      .fillna(0, ["hits"])
      .select(F.avg(F.col("hits") / F.col("total")))
      .first()
  )
  recall = float(row[0] or 0.0)

  # Pull current hyperparameter values and log validation Recall@10.
  r, mi, rp = p[als_spec.rank], p[als_spec.maxIter], p[als_spec.regParam]
  print(f"rank={r}, maxIter={mi}, regParam={rp} -> val Recall@10: {recall:.4f}")

  # Update best hyperparameters if this combination beats the current best Recall@10.
  if recall > best_recall:
    best_recall, best_rank, best_maxIter, best_regParam = recall, r, mi, rp

# Print total elapsed time once all grid combinations finish.
print(f"\nGrid search time: {time.time() - t0:.2f}s")

# Display the winning hyperparameters and validation Recall@10 for the next cell.
print(
    f"\nBest -> rank={best_rank}, maxIter={best_maxIter},"
    f" regParam={best_regParam}, val Recall@10={best_recall:.4f}"
)

rank=10, maxIter=10, regParam=0.01 -> val Recall@10: 0.0237


rank=10, maxIter=10, regParam=0.1 -> val Recall@10: 0.0251


rank=10, maxIter=20, regParam=0.01 -> val Recall@10: 0.0249


rank=10, maxIter=20, regParam=0.1 -> val Recall@10: 0.0260


rank=20, maxIter=10, regParam=0.01 -> val Recall@10: 0.0215


rank=20, maxIter=10, regParam=0.1 -> val Recall@10: 0.0229


... (3 outputs left out)

rank=40, maxIter=10, regParam=0.1 -> val Recall@10: 0.0197


rank=40, maxIter=20, regParam=0.01 -> val Recall@10: 0.0189


[Stage 32523:==================================================> (98 + 2) / 100]

rank=40, maxIter=20, regParam=0.1 -> val Recall@10: 0.0205

Grid search time: 423.28s

Best -> rank=10, maxIter=20, regParam=0.1, val Recall@10=0.0260


**Training time experience.**  
With 32 cores and 256GB RAM, this ran in ~7 minutes.

**Best values.**  

The above cell should return these results:

```text
Best -> rank=10, maxIter=20, regParam=0.1, val Recall@10=0.0260

In [23]:
# Retrain the best configuration on train + validation data.

# Combine trainDF and valDF.
tvDF = trainDF.union(valDF)
tvDF = tvDF.cache()

# Update the model spec to use the best config.
als_spec_tuned = ALS(
    userCol="userid",
    itemCol="artistid",
    ratingCol="playcount",
    implicitPrefs=True,
    coldStartStrategy="drop",
    rank=best_rank,
    maxIter=best_maxIter,
    regParam=best_regParam,
    alpha=0.01
)

print("The tuned als_spec has been built.")

The tuned als_spec has been built.


In [24]:
# THIS STEP TAKES A FEW SECONDS...PUT INTO ITS OWN CELL IN CASE IT RUNS LONG LIKE THE CV MODEL TUNING.

# Time this step.
t0 = time.time()

# Fit the final model on the combined train and validation data.
als_model_tuned = als_spec_tuned.fit(tvDF) 

# Show how much elapsed.
print("train time:", time.time() - t0)

print("The final tuned model has been fit using the training and validation data.")

train time: 5.79625391960144
The final tuned model has been fit using the training and validation data.


**Training time experience.**  
With 32 cores and 256GB RAM, this code tends to run around 5 seconds.  

In [25]:
# TASK 7 (1 PT)
# Calculate Recall@K on the test set and report the value.

# For this, I will skip trying to integrate SQL and instead use PySpark.
# I had to get help from Gemini because I struggled a lot to piece this together.

# Import the necessary functions library.
from pyspark.sql import functions as F

# Get unique users and unique (userid, artistid) ground-truth pairs in testDF.
test_users = testDF.select("userid").distinct()
actual_test = testDF.select("userid", "artistid").distinct()

# Generate top-10 recommendations for test users and explode into (userid, artistid) rows.
test_top10_recs = als_model_tuned.recommendForUserSubset(test_users, 10)
test_top10DF = (
    test_top10_recs.select(
        F.col("userid"),
        F.explode(F.col("recommendations")).alias("rec"),
    )
    .select(
        F.col("userid"),
        F.col("rec.artistid").alias("artistid"),
    )
)

# STEP 1: DENOMINATOR.
# Count unique artists per user in testDF.
all_counts = actual_test.groupBy("userid").agg(F.count("*").alias("total_actual"))

# STEP 2: NUMERATOR.
# Count true positives (hits) where a recommended top-10 artist appears in testDF for that user.
tp_all_count = (
    test_top10DF.join(actual_test, on=["userid", "artistid"], how="inner")
    .groupBy("userid")
    .agg(F.count("*").alias("tp_count"))
)

# STEP 3: CALCULATE Recall@K.
recall_df = (
    all_counts.join(tp_all_count, on="userid", how="left")
    .na.fill(0, subset=["tp_count"])
    .withColumn("recall_k", F.col("tp_count") / F.col("total_actual"))
)

# Average Recall@10 across all test users.
row = recall_df.agg(F.avg("recall_k")).first()
mean_recall_k = float(row[0]) if row and row[0] is not None else 0.0

print(f"Mean Recall@K across the test set is: {mean_recall_k:.4f}")

Mean Recall@K across the test set is: 0.0284


**Test set Recall@K.**  
The prior cell should return something like this: *Mean Recall@K across the test set is: 0.0284*
This is better than the Recall@K obtained for userid=536 (0.0).

In [26]:
# TASK 8 (1 PT)
# Calculate Recall@K for Test User 1 
# Justify the result, showing the recommendations and user activity. Did hyperparameter tuning seem to help for this user?

# For this, I will skip trying to integrate SQL and instead use PySpark.

# Import the necessary functions library.
from pyspark.sql import functions as F

# Get all the recommendations for userid = 536.
target_user = 536
user536_recs = spark.createDataFrame([(target_user,)], ["userid"])

# Generate the top 10 recommendations from this subset.
user1_recs_top10 = als_model_tuned.recommendForUserSubset(user536_recs, 10)

# Explode recommendations into a flat DataFrame of predicted items.
top10DF = user1_recs_top10.select(
    F.col("userid"),
    F.explode(F.col("recommendations")).alias("rec")
).select(
    F.col("userid"),
    F.col("rec.artistid").alias("artistid")
)

# STEP 1: DENOMINATOR.
# Get all rows for userid = 536 from test data.
testDF_user536 = testDF.filter(F.col("userid") == target_user).select("userid", "artistid").distinct()
testDF_user536_count = testDF_user536.count()

# STEP 2: NUMERATOR
# Count True Positives (tp) using an inner join.
# Get the total of these that were in the predicted set; these are the true positives (tp).
tp_count = top10DF.join(testDF_user536, on=["userid", "artistid"], how="inner").count()

# STEP 3: CALCULATE the Recall@K.
recall_k = tp_count / testDF_user536_count if testDF_user536_count > 0 else 0.0

print(f"The Recall@K for userid={target_user} (tuned model) is: {recall_k:.4f} ({tp_count}/{testDF_user536_count})")

The Recall@K for userid=536 (tuned model) is: 0.0000 (0/13)


In [28]:
# Show top-10 recommended artists for userid = 536.
print(f"Top-10 recommendations for userid={target_user}:")
top10DF.show(10, truncate=False)

# Inspect user activity across trainDF, valDF, and testDF.
print(f"Activity counts for userid={target_user}:")
print("  trainDF rows:", trainDF.filter(F.col("userid") == target_user).count())
print("  valDF rows:  ", valDF.filter(F.col("userid") == target_user).count())
print("  testDF rows: ", testDF.filter(F.col("userid") == target_user).count())

# Show actual test set rows and unique artists listened to for userid = 536.
testDF_user536 = testDF.filter(F.col("userid") == target_user)
actual_user536 = testDF_user536.select("userid", "artistid").distinct()
testDF_user536_count = actual_user536.count()

print(f"--- Actual Test Set Artists Listened To for User {target_user} ---")
actual_user536.show(truncate=False)

# Compare against an untuned baseline ALS model fit on tvDF.
baseline_model = als_spec.fit(tvDF)
base_top10DF = (
    baseline_model.recommendForUserSubset(user536_recs, 10)
    .select(
        F.col("userid"),
        F.explode(F.col("recommendations")).alias("rec"),
    )
    .select(
        F.col("userid"),
        F.col("rec.artistid").alias("artistid"),
    )
)
base_tp = base_top10DF.join(actual_user536, on=["userid", "artistid"], how="inner").count()
base_recall_k = base_tp / testDF_user536_count if testDF_user536_count > 0 else 0.0
print(f"The Recall@K for userid={target_user} (untuned baseline) is: {base_recall_k:.4f} ({base_tp}/{testDF_user536_count})")

Top-10 recommendations for userid=536:
+------+--------+
|userid|artistid|
+------+--------+
|536   |82      |
|536   |1000113 |
|536   |979     |
|536   |1394    |
|536   |313     |
|536   |793     |
|536   |976     |
|536   |606     |
|536   |1223    |
|536   |189     |
+------+--------+

Activity counts for userid=536:
  trainDF rows: 47
  valDF rows:   9
  testDF rows:  13
--- Actual Test Set Artists Listened To for User 536 ---
+------+--------+
|userid|artistid|
+------+--------+
|536   |1205    |
|536   |1598    |
|536   |4449    |
|536   |1012701 |
|536   |1025289 |
|536   |1037206 |
|536   |1173467 |
|536   |1235135 |
|536   |1339526 |
|536   |2040459 |
|536   |10378766|
|536   |10380310|
|536   |10565309|
+------+--------+

The Recall@K for userid=536 (untuned baseline) is: 0.0000 (0/13)


**Conclusion.**
The results above suggest the model did not recommended any one of the artists that userid=536 actually listened to.
Reviewing the top 10 recommendations and the artists in the test data set confirms this.
Per output above (visual inspection) there is just no overlap between the predictions and the test data for userid=536.

Thus, the hyperparameter tuning did NOT improve recommendations for this user.